# 📦 Notebook 01 — Data Loading & Schema Validation
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the PubMedQA dataset from HuggingFace
- Validate schema: confirm `question`, `context` (list), `long_answer`, `final_decision` columns exist
- Log raw row count, null counts, and a 5-row sample
- Save raw data locally as `data/raw/pubmedqa_raw.csv`
- Generate `reports/schema_validation_report.md`

### 📋 Deliverables
- `data/raw/pubmedqa_raw.csv`
- `reports/schema_validation_report.md`

---

## 1. Imports & Configuration

In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DATA_DIR = '../data/raw'
REPORTS_DIR = '../reports'
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(RAW_DATA_DIR, 'pubmedqa_raw.csv')
REPORT_PATH = os.path.join(REPORTS_DIR, 'schema_validation_report.md')

# ── Dataset config ─────────────────────────────────────────────────────────
DATASET_NAME = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_artificial'
EXPECTED_COLUMNS = ['pubid', 'question', 'context', 'long_answer', 'final_decision']
EXPECTED_ROW_COUNT = 211_000

print('✅ Imports successful')
print(f'📁 Raw data → {OUTPUT_PATH}')
print(f'📄 Report  → {REPORT_PATH}')

✅ Imports successful
📁 Raw data → ../data/raw\pubmedqa_raw.csv
📄 Report  → ../reports\schema_validation_report.md


## 2. Load Dataset from HuggingFace

In [2]:
print(f'⏳ Loading dataset: {DATASET_NAME} ({DATASET_SUBSET}) ...')

dataset = load_dataset(DATASET_NAME, DATASET_SUBSET)

print('\n✅ Dataset loaded successfully!')
print('\n📊 Dataset overview:')
print(dataset)

⏳ Loading dataset: qiaojin/PubMedQA (pqa_artificial) ...

✅ Dataset loaded successfully!

📊 Dataset overview:
DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 211269
    })
})


## 3. Inspect Available Splits

In [3]:
print('🔍 Available splits:', list(dataset.keys()))
print()

for split_name, split_data in dataset.items():
    print(f'  Split: "{split_name}" → {len(split_data):,} records')

🔍 Available splits: ['train']

  Split: "train" → 211,269 records


## 4. Convert to Pandas DataFrame

In [4]:
main_split = list(dataset.keys())[0]
print(f'📌 Using split: "{main_split}"')

df = pd.DataFrame(dataset[main_split])

print(f'\n✅ Converted to DataFrame')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

📌 Using split: "train"

✅ Converted to DataFrame
   Shape: 211,269 rows × 5 columns


## 5. Schema Validation

In [5]:
print('🔍 Schema Validation')
print('─' * 50)

# Check columns
actual_columns = df.columns.tolist()
missing_cols = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_cols = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

print(f'Expected columns: {EXPECTED_COLUMNS}')
print(f'Actual columns:   {actual_columns}')
print(f'Missing columns:  {missing_cols if missing_cols else "None"}')
print(f'Extra columns:    {extra_cols if extra_cols else "None"}')

if not missing_cols and not extra_cols:
    schema_status = f'PASSED — all {len(EXPECTED_COLUMNS)} expected columns present, no drift'
    print(f'\n✅ {schema_status}')
else:
    schema_status = f'FAILED — Missing: {missing_cols}, Extra: {extra_cols}'
    print(f'\n❌ {schema_status}')

# Check row count
print(f'\nExpected ~{EXPECTED_ROW_COUNT:,} rows, got {len(df):,} rows')
if len(df) >= EXPECTED_ROW_COUNT:
    print('✅ Row count OK')
else:
    print(f'⚠️  Fewer rows than expected')

# Check data types
print(f'\n📋 Data Types:')
print(df.dtypes)

🔍 Schema Validation
──────────────────────────────────────────────────
Expected columns: ['pubid', 'question', 'context', 'long_answer', 'final_decision']
Actual columns:   ['pubid', 'question', 'context', 'long_answer', 'final_decision']
Missing columns:  None
Extra columns:    None

✅ PASSED — all 5 expected columns present, no drift

Expected ~211,000 rows, got 211,269 rows
✅ Row count OK

📋 Data Types:
pubid              int64
question          object
context           object
long_answer       object
final_decision    object
dtype: object


## 6. Null Counts

In [6]:
print('🔎 Missing Values per Column:')
print('─' * 40)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df)
print()

if missing.sum() == 0:
    null_status = 'No missing values found'
    print(f'✅ {null_status}')
else:
    null_status = f'{missing.sum():,} total missing values'
    print(f'⚠️  {null_status}')

🔎 Missing Values per Column:
────────────────────────────────────────
                Missing Count  Missing %
pubid                       0        0.0
question                    0        0.0
context                     0        0.0
long_answer                 0        0.0
final_decision              0        0.0

✅ No missing values found


## 7. Duplicate Check

In [7]:
# Convert to string to handle unhashable types (e.g. list-of-dict in context)
n_duplicates = df.astype(str).duplicated().sum()
print(f'🔎 Duplicate rows: {n_duplicates:,}')

if n_duplicates == 0:
    dup_status = 'No duplicates found'
    print(f'✅ {dup_status}')
else:
    dup_status = f'{n_duplicates} duplicate rows found'
    print(f'⚠️  {dup_status}')

🔎 Duplicate rows: 0
✅ No duplicates found


## 8. 5-Row Sample

In [8]:
print('👀 First 5 records:')
pd.set_option('display.max_colwidth', 150)
df.head(5)


👀 First 5 records:


,pubid,question,context,long_answer,final_decision
0,25429730,Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?,{'contexts': ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) repre...,"As ILC2s are elevated in patients with CRSwNP, they may drive nasal polyp formation in CRS. ILC2s are also linked with high tissue and blood eosin...",yes
1,25433161,Does vagus nerve contribute to the development of steatohepatitis and obesity in phosphatidylethanolamine N-methyltransferase deficient mice?,"{'contexts': ['Phosphatidylethanolamine N-methyltransferase (PEMT), a liver enriched enzyme, is responsible for approximately one third of hepatic...",Neuronal signals via the hepatic vagus nerve contribute to the development of steatohepatitis and protection against obesity in HFD fed Pemt(-/-) ...,yes
2,25445714,Does psammaplin A induce Sirtuin 1-dependent autophagic cell death in doxorubicin-resistant MCF-7/adr human breast cancer cells and xenografts?,"{'contexts': ['Psammaplin A (PsA) is a natural product isolated from marine sponges, which has been demonstrated to have anticancer activity again...","PsA significantly inhibited MCF-7/adr cells proliferation in a concentration-dependent manner, with accumulation of cells in G2/M phase of the cel...",yes
3,25431941,Is methylation of the FGFR2 gene associated with high birth weight centile in humans?,"{'contexts': ['This study examined links between DNA methylation and birth weight centile (BWC), and explored the impact of genetic variation.', '...",We identified a novel biologically plausible candidate (FGFR2) for with BWC that merits further study.,yes
4,25432519,Do tumor-infiltrating immune cell profiles and their change after neoadjuvant chemotherapy predict response and prognosis of breast cancer?,{'contexts': ['Tumor microenvironment immunity is associated with breast cancer outcome. A high lymphocytic infiltration has been associated with ...,"Breast cancer immune cell subpopulation profiles, determined by immunohistochemistry-based computerized analysis, identify groups of patients char...",yes


## 9. Basic Text Statistics

In [9]:
text_cols = df.select_dtypes(include='object').columns.tolist()

print('📏 Text Length Statistics (character count):')
print('─' * 50)

for col in text_cols:
    lengths = df[col].astype(str).str.len()
    print(f'\n[{col}]')
    print(f'  Min:    {lengths.min():,}')
    print(f'  Max:    {lengths.max():,}')
    print(f'  Mean:   {lengths.mean():,.0f}')
    print(f'  Median: {lengths.median():,.0f}')

📏 Text Length Statistics (character count):
──────────────────────────────────────────────────

[question]
  Min:    17
  Max:    664
  Mean:   114
  Median: 112

[context]
  Min:    72
  Max:    5,869
  Mean:   1,732
  Median: 1,731

[long_answer]
  Min:    1
  Max:    3,340
  Mean:   260
  Median: 238

[final_decision]
  Min:    2
  Max:    3
  Mean:   3
  Median: 3


## 10. Save Raw Data

In [10]:
df.to_csv(OUTPUT_PATH, index=False)

file_size = os.path.getsize(OUTPUT_PATH) / 1024**2

print(f'✅ Raw dataset saved to: {OUTPUT_PATH}')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.shape[1]}')
print(f'   File size: {file_size:.1f} MB')

✅ Raw dataset saved to: ../data/raw\pubmedqa_raw.csv
   Rows: 211,269
   Columns: 5
   File size: 428.9 MB


In [15]:
# ── Auto-upload to HuggingFace ────────────────────────────────────────────
import sys
sys.path.append(os.path.abspath('..'))

from src.data.hub import upload_all_data

print("\n📤 Syncing to HuggingFace...")
upload_all_data()


📤 Syncing to HuggingFace...
📤 Uploading data to: huggingface.co/datasets/AbdoMatrix/healthcare-rag-data
────────────────────────────────────────────────────────────
  ⚠️  HF_TOKEN not set.
     Set the HF_TOKEN env var in your .env file or run:
     export HF_TOKEN=hf_xxxxx

⚠️  No upload token — skipping all files.


{'uploaded': 0, 'skipped': 5, 'failed': 0}

## 11. Generate Schema Validation Report

In [12]:
sample_md = df.head(5).to_markdown(index=False)

report = f"""# Schema Validation Report
**Healthcare RAG-Powered Medical Q&A Assistant**

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Dataset Info
| Item | Value |
|---|---|
| Source | `{DATASET_NAME}` |
| Split | `{main_split}` |
| Row count | {len(df):,} |
| Expected rows | ~{EXPECTED_ROW_COUNT:,} |

## Schema Validation
| Item | Value |
|---|---|
| Expected columns | {EXPECTED_COLUMNS} |
| Actual columns | {actual_columns} |
| Missing columns | {missing_cols if missing_cols else 'None'} |
| Extra columns | {extra_cols if extra_cols else 'None'} |
| **Status** | **{schema_status}** |

## Null Counts
{missing_df.to_markdown()}

**Status:** {null_status}

## Duplicates
| Item | Value |
|---|---|
| Duplicate rows | {n_duplicates} |
| **Status** | **{dup_status}** |

## 5-Row Sample
{sample_md}
"""

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(f'✅ Schema validation report saved to: {REPORT_PATH}')

✅ Schema validation report saved to: ../reports\schema_validation_report.md


## ✅ Summary

| Item | Result |
|---|---|
| Dataset | `qiaojin/PubMedQA` (`pqa_artificial` subset) |
| Records | {len(df):,} |
| Dataset Subset | `{DATASET_SUBSET}` |
| Columns | {actual_columns} |
| Schema | {schema_status} |
| Missing values | {null_status} |
| Duplicates | {dup_status} |
| Saved raw CSV | `data/raw/pubmedqa_raw.csv` |
| Saved report | `reports/schema_validation_report.md` |

---

### ➡️ Next Step
Open **`02_preprocessing.ipynb`** to clean and normalise the dataset.